# 04 Collect Yearly Data

Collect yearly pathogen–antibiotic candidate counts and background marginals from PubMed.
This stage writes the candidate parquet used by abstract preparation, cost estimation, and prediction notebooks.

## Inputs

- Pipeline settings from `outputs/pipeline_config.yaml`.
- Pathogen and antibiotic synonym tables from `assets/`.
- PubMed search results, metadata, and interval caches already on disk are reused when available.
- After changing the PubMed query, set `collection.refresh_cached_intervals: true` to recheck cached pair/date windows for the full configured pathogen–antibiotic set. Set `collection.refresh_cached_pairs` to a list of `[pathogen, antibiotic]` pairs only when intentionally limiting a refresh. Existing PMID metadata is reused; new PMID metadata is fetched and added. Set the refresh option back to `false` after completion.

## Outputs

- `candidate_amr_pair_year_counts.parquet` and `candidate_amr_pair_pmid_links.parquet`.
- Pathogen and antibiotic background-count parquets.
- PubMed interval and metadata caches used to resume collection.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import build_and_print_collection_preflight
from amr.pipelines.collection import (
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)
OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR

PATHOGENS_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
PATHOGEN_PRIORITY_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
MIN_PATHOGENIC_INDEX = cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index")
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
OUTPUT_YEARLY_AMR_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
OUTPUT_YEARLY_SCREEN_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
OUTPUT_UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
OUTPUT_SYSTEM1_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.system1_decisions", PROJECT_ROOT)
OUTPUT_PATHOGEN_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_background", PROJECT_ROOT)
OUTPUT_ANTIBIOTIC_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_background", PROJECT_ROOT)
INTERVAL_PMIDS_PATH = cfg_path(PIPELINE_CONFIG, "paths.interval_pmids", PROJECT_ROOT)
YEARLY_PAIR_PMID_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_pair_pmids", PROJECT_ROOT)
PUBMED_METADATA_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.pubmed_metadata_cache", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
GROUP_SEARCH_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.group_search_cache", PROJECT_ROOT)
PUBMED_EMAIL = cfg(PIPELINE_CONFIG, "collection.pubmed_email")

# Date windows: TOTAL_* defines the canonical range; SLICE_* limits this run for partitioned/resumable work.
TOTAL_START_YEAR = cfg(PIPELINE_CONFIG, "collection.total_start_year")
TOTAL_END_YEAR = cfg(PIPELINE_CONFIG, "collection.total_end_year")
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_start_year")
SLICE_END_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_end_year")
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = cfg(PIPELINE_CONFIG, "collection.interval_year_gap")

# Vocabulary limits: use None for full canonical runs; integers are useful for quick/resumable subsets.
NUM_PATHOGENS = cfg(PIPELINE_CONFIG, "collection.num_pathogens")
NUM_ANTIBIOTICS = cfg(PIPELINE_CONFIG, "collection.num_antibiotics")
EXTRA_AMR_KEYWORDS = cfg(PIPELINE_CONFIG, "collection.extra_amr_keywords", [])

# Select which saved prediction source builds the screened dataset.
ACTIVE_PREDICTION_SOURCE = str(cfg(PIPELINE_CONFIG, "screening.active_source", "llm")).lower()
SYSTEM1_PROBABILITY_THRESHOLD = float(cfg(PIPELINE_CONFIG, "screening.system1_probability_threshold", 0.5))
SYSTEM1_MODEL = cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13")
SYSTEM1_RUN_ID = str(cfg(PIPELINE_CONFIG, "system1.run_id", "jev-default"))
SYSTEM1_RUN_FILTER = bool(cfg(PIPELINE_CONFIG, "system1.run_filter", True))
SYSTEM1_MAX_CALLS = cfg(PIPELINE_CONFIG, "system1.max_calls")
SYSTEM1_RATE_LIMIT = float(cfg(PIPELINE_CONFIG, "system1.rate_limit", 0.0))
SYSTEM1_SAVE_EVERY = int(cfg(PIPELINE_CONFIG, "system1.save_every", 100))
SYSTEM1_PROGRESS_EVERY = int(cfg(PIPELINE_CONFIG, "system1.progress_every", 100))
if ACTIVE_PREDICTION_SOURCE not in {"llm", "system1"}:
    raise ValueError("screening.active_source must be 'llm' or 'system1'.")
if not 0.0 <= SYSTEM1_PROBABILITY_THRESHOLD <= 1.0:
    raise ValueError("screening.system1_probability_threshold must be between 0 and 1.")

# LLM decision modes: rerun_unified_decisions=True ignores active decisions but reuses cached abstracts.
RUN_LLM_FILTER = cfg(PIPELINE_CONFIG, "llm.run_filter")
RERUN_UNIFIED_LLM_DECISIONS = cfg(PIPELINE_CONFIG, "llm.rerun_unified_decisions")
MAX_LLM_CALLS = cfg(PIPELINE_CONFIG, "llm.max_calls")
LLM_MAX_WORKERS = cfg(PIPELINE_CONFIG, "llm.max_workers")
LLM_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.batch_size")
FAILED_RETRY_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_failures", PROJECT_ROOT)
MAX_FAILURE_RETRIES = cfg(PIPELINE_CONFIG, "llm.max_failure_retries")
ABSTRACT_FETCH_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.abstract_fetch_batch_size")
PREFETCH_ABSTRACTS = cfg(PIPELINE_CONFIG, "llm.prefetch_abstracts")
LLM_CATEGORY_BACKFILL_DECISIONS = cfg(PIPELINE_CONFIG, "llm.category_backfill_decisions")
LLM_RETRY_CACHED_FAILURES = bool(cfg(PIPELINE_CONFIG, "llm.retry_cached_failures", False))
LLM_INCLUDE_EXPLANATORY_TEXT = bool(cfg(PIPELINE_CONFIG, "llm.include_explanatory_text", False))
LLM_REASONING_EFFORT = cfg(PIPELINE_CONFIG, "llm.reasoning_effort", "minimal")
UNIFIED_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "llm.progress_every")
SCREENED_INCLUDE_DECISIONS = tuple(cfg(PIPELINE_CONFIG, "llm.screened_include_decisions", ["strict", "yes"]))

# PubMed throughput and background-count controls.
COLLECTION_VERBOSE = cfg(PIPELINE_CONFIG, "collection.collection_verbose")
COLLECTION_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "collection.collection_progress_every")
PAIR_FULL_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.pair_full_fetch_threshold")
COLLECTION_SEARCH_MODE = cfg(PIPELINE_CONFIG, "collection.search_mode", "grouped")
COLLECTION_ANTIBIOTIC_GROUP_SIZE = int(cfg(PIPELINE_CONFIG, "collection.antibiotic_group_size", 10))
COLLECTION_SAVE_EVERY = cfg(PIPELINE_CONFIG, "collection.save_every", 500)
COLLECTION_SUMMARY_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.summary_batch_size", 1000)
COLLECTION_RETRIEVE_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.retrieve_batch_size", 9000)
COLLECTION_MAX_WORKERS = cfg(PIPELINE_CONFIG, "collection.max_workers", 4)
COLLECTION_MAX_PENDING_FUTURES = cfg(PIPELINE_CONFIG, "collection.max_pending_futures", 8)
COLLECTION_PARTIAL_INTERVAL_MAX = cfg(PIPELINE_CONFIG, "collection.partial_interval_max", 1)
COLLECTION_REFRESH_CACHED_INTERVALS = bool(cfg(PIPELINE_CONFIG, "collection.refresh_cached_intervals", False))
COLLECTION_REFRESH_CACHED_PAIRS = tuple(tuple(pair) for pair in cfg(PIPELINE_CONFIG, "collection.refresh_cached_pairs", []))
BACKGROUND_SCOPE = cfg(PIPELINE_CONFIG, "collection.background_scope")
BACKGROUND_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.background_fetch_threshold")
BACKGROUND_METADATA_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.background_metadata_batch_size")


## Load Prepared Pathogen and Antibiotic Inputs



In [4]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_csv(ANTIBIOTIC_COARSENING_PATH)
antibiotic_synonym_df = pd.read_csv(ANTIBIOTIC_SYNONYM_PATH)

if not PATHOGEN_PRIORITY_PATH.exists():
    raise FileNotFoundError(
        f"Pathogen priority file not found: {PATHOGEN_PRIORITY_PATH}. "
        "Run notebooks/01_prepare_inputs.ipynb first."
    )

pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_PATH)
required_priority_cols = {"CanonicalSpecies", "pathogenic_index"}
missing_priority_cols = required_priority_cols - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_PATH} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"],
    errors="coerce",
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[
    pathogen_priority_df["pathogenic_index"] > float(MIN_PATHOGENIC_INDEX)
]
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"],
    ascending=[False, True],
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")

all_pathogens = pathogen_priority_df["CanonicalSpecies"].tolist()
all_antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

pathogens = all_pathogens if NUM_PATHOGENS is None else all_pathogens[: int(NUM_PATHOGENS)]
antibiotics = all_antibiotics if NUM_ANTIBIOTICS is None else all_antibiotics[: int(NUM_ANTIBIOTICS)]

print(f"Pathogen priority source: {PATHOGEN_PRIORITY_PATH}")
print(f"Full synonym lookup vocabulary: {bacteria_synonym_df['bacteria name'].nunique():,}")
print(
    f"Pathogenic-index pathogen vocabulary: {len(all_pathogens):,} "
    f"(MIN_PATHOGENIC_INDEX>{MIN_PATHOGENIC_INDEX})"
)
print(f"Antibiotic coarsening source: {ANTIBIOTIC_COARSENING_PATH}")
print(f"Antibiotic synonym source: {ANTIBIOTIC_SYNONYM_PATH} ({len(antibiotic_synonym_df):,} rows)")
print(f"Full antibiotic vocabulary: {len(all_antibiotics):,}")
print(f"Active pathogens this run: {len(pathogens):,} (NUM_PATHOGENS={NUM_PATHOGENS})")
print(f"Active antibiotics this run: {len(antibiotics):,} (NUM_ANTIBIOTICS={NUM_ANTIBIOTICS})")
print("Top active pathogens by pathogenic_index:")
display(
    pathogen_priority_df[["CanonicalSpecies", "pathogenic_index", "n_causation", "n_articles"]]
    .head(len(pathogens))
    .reset_index(drop=True)
)



Pathogen priority source: /Users/f.costa/Code/Antimicrobial-Resistance/assets/counts_extended2.csv
Full synonym lookup vocabulary: 19,956
Pathogenic-index pathogen vocabulary: 915 (MIN_PATHOGENIC_INDEX>0.0)
Antibiotic coarsening source: /Users/f.costa/Code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv
Antibiotic synonym source: /Users/f.costa/Code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv (422 rows)
Full antibiotic vocabulary: 245
Active pathogens this run: 915 (NUM_PATHOGENS=None)
Active antibiotics this run: 245 (NUM_ANTIBIOTICS=None)
Top active pathogens by pathogenic_index:


,CanonicalSpecies,pathogenic_index,n_causation,n_articles
0,Salmonella enterica,1.000000,554,24694
1,Mycobacterium tuberculosis,0.928144,299,20559
2,Clostridioides difficile,0.910833,298,17012
3,Haemophilus influenzae,0.860892,200,14537
4,Klebsiella pneumoniae,0.857160,176,15847
...,...,...,...,...
910,Ureibacillus massiliensis,0.062260,2,4
911,Vandammella animalimorsus,0.062260,2,3
912,Virgibacillus pantothenticus,0.062260,2,2
913,Waddlia chondrophila,0.062260,2,70


## Collect Yearly Pair Dataset and Background Marginals


In the next code cell, the notebook will automatically gather research paper IDs from PubMed for each pathogen-antibiotic combination in your selected year range, convert those papers into yearly counts, and then save the results to local files so future runs can continue from where you left off. It also collects yearly background publication totals for each pathogen and antibiotic, which are later used to calculate scientific interest in a fair, comparable way.


## Resume Preflight

Estimate how much collection work is already cached on disk before the resumable collection step starts.


In [5]:
preflight = build_and_print_collection_preflight(
    pathogens=pathogens,
    antibiotics=antibiotics,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_path=YEARLY_PAIR_PMID_PATH,
    yearly_output_path=OUTPUT_YEARLY_AMR_PATH,
    pathogen_bg_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_bg_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    background_start_year=SLICE_START_YEAR,
    background_end_year=SLICE_END_YEAR,
    total_min_date=TOTAL_MIN_DATE,
    total_max_date=TOTAL_MAX_DATE,
    total_background_start_year=TOTAL_START_YEAR,
    total_background_end_year=TOTAL_END_YEAR,
)


COLLECTION PREFLIGHT

Existing yearly AMR span on disk: None .. None
Existing yearly AMR rows on disk: 0
Existing yearly pair PMID rows on disk: 0

TOTAL WINDOW SUMMARY

Date window: 1970/01/01 .. 2026/12/31
Intervals in window: 12 -> [('1970/01/01', '1974/12/31'), ('1975/01/01', '1979/12/31'), ('1980/01/01', '1984/12/31'), ('1985/01/01', '1989/12/31'), ('1990/01/01', '1994/12/31'), ('1995/01/01', '1999/12/31'), ('2000/01/01', '2004/12/31'), ('2005/01/01', '2009/12/31'), ('2010/01/01', '2014/12/31'), ('2015/01/01', '2019/12/31'), ('2020/01/01', '2024/12/31'), ('2025/01/01', '2026/12/31')]
Interval cache coverage: 0 / 2,690,100 rows (0.00%)
Missing interval rows still to collect: 2,690,100
Cached interval breadth: 0 pathogens, 0 antibiotics, 0 unique pairs
Existing yearly AMR rows in window: 0
Existing yearly pair PMID rows in window: 0
Pathogen background cache: 10,854 / 52,155 rows (20.81%)
Missing pathogen background rows: 41,301 (79.19%)
Antibiotic background cache: 5,346 / 13,965 r

In [ ]:
print(
    f"PubMed search mode={COLLECTION_SEARCH_MODE}; antibiotic group size={COLLECTION_ANTIBIOTIC_GROUP_SIZE}; "
    f"refresh cached intervals={COLLECTION_REFRESH_CACHED_INTERVALS}."
)

yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
    full_fetch_threshold=PAIR_FULL_FETCH_THRESHOLD,
    save_every=COLLECTION_SAVE_EVERY,
    summary_batch_size=COLLECTION_SUMMARY_BATCH_SIZE,
    retrieve_batch_size=COLLECTION_RETRIEVE_BATCH_SIZE,
    max_workers=COLLECTION_MAX_WORKERS,
    max_pending_futures=COLLECTION_MAX_PENDING_FUTURES,
    partial_interval_max=COLLECTION_PARTIAL_INTERVAL_MAX,
    search_mode=COLLECTION_SEARCH_MODE,
    antibiotic_group_size=COLLECTION_ANTIBIOTIC_GROUP_SIZE,
    group_search_cache_path=GROUP_SEARCH_CACHE_PATH,
    abstract_cache_path=ABSTRACT_CACHE_PATH,
    refresh_cached_intervals=COLLECTION_REFRESH_CACHED_INTERVALS,
    refresh_cached_pairs=COLLECTION_REFRESH_CACHED_PAIRS,
    collection_verbose=COLLECTION_VERBOSE,
    collection_progress_every=COLLECTION_PROGRESS_EVERY,
)


PubMed search mode=grouped; antibiotic group size=10; refresh cached intervals=True.


In [ ]:
if BACKGROUND_SCOPE == "observed_entities":
    background_pathogens = sorted(yearly_amr_df["pathogen"].dropna().astype(str).unique())
    background_antibiotics = sorted(yearly_amr_df["antibiotic"].dropna().astype(str).unique())
elif BACKGROUND_SCOPE == "all_entities":
    background_pathogens = pathogens
    background_antibiotics = antibiotics
else:
    raise ValueError("BACKGROUND_SCOPE must be 'observed_entities' or 'all_entities'.")

print(
    f"Background scope={BACKGROUND_SCOPE}: "
    f"{len(background_pathogens):,} pathogens, {len(background_antibiotics):,} antibiotics"
)

pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=background_pathogens,
    antibiotics=background_antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    fetch_threshold=BACKGROUND_FETCH_THRESHOLD,
    metadata_batch_size=BACKGROUND_METADATA_BATCH_SIZE,
)

print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"yearly pair PMID rows: {len(yearly_pair_pmid_df):,}")
print(f"newly enriched pair PMID rows this run: {len(enriched_pair_pmid_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")
